<a href="https://colab.research.google.com/github/EdsonCaldas/Multi_currency_billing_pipeline/blob/main/Multi_currency_billing_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import time
import requests
import pandas as pd
from datetime import datetime
from urllib3.util.retry import Retry
from requests.adapters import HTTPAdapter

# Cache global para reaproveitar cotações já buscadas na mesma execução
CACHE_TAXAS = {}

# ==========================================
# 1. MÓDULO DE CONVERSÃO DE CÂMBIO (COM RETRY E CACHE)
# ==========================================
def criar_sessao_com_retry():
    """
    Cria uma sessão HTTP com espera progressiva para evitar bloqueios 429.
    """
    session = requests.Session()
    retry_strategy = Retry(
        total=3,                # Tenta até 3 vezes
        backoff_factor=3,       # Aguarda 3s, 6s, 12s entre as tentativas
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"]
    )
    adapter = HTTPAdapter(max_retries=retry_strategy)
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    return session

def obter_taxa_cambio(moeda_origem: str, moeda_destino: str = "BRL") -> float:
    """
    Busca a taxa de câmbio em tempo real com proteção contra erro 429.
    """
    if moeda_origem.upper() == moeda_destino.upper():
        return 1.0

    par = f"{moeda_origem.upper()}-{moeda_destino.upper()}"
    chave = f"{moeda_origem.upper()}{moeda_destino.upper()}"

    # Retorna do Cache se já consultou nesta execução
    if chave in CACHE_TAXAS:
        return CACHE_TAXAS[chave]

    # URL 100% limpa e sem asteriscos
    url = f"https://economia.awesomeapi.com.br/last/{par}"

    try:
        session = criar_sessao_com_retry()
        resposta = session.get(url, timeout=10)
        resposta.raise_for_status()

        dados = resposta.json()
        taxa = float(dados[chave]["bid"])

        # Guarda no cache
        CACHE_TAXAS[chave] = taxa
        return taxa

    except Exception as e:
        print(f"⚠️ Erro ao buscar câmbio para {par}: {e}. Usando taxa fallback 1.0.")
        return 1.0

# ==========================================
# 2. ESTEIRA DE FATURAMENTO E CONCILIAÇÃO
# ==========================================
class EsteiraFaturamento:
    def __init__(self, moeda_base: str = "BRL"):
        self.moeda_base = moeda_base
        self.faturas = []

    def adicionar_fatura(self, id_fatura: str, cliente: str, valor_original: float, moeda: str, valor_recebido: float = 0.0):
        """
        Cadastra e processa a conciliação de uma fatura internacional.
        """
        taxa = obter_taxa_cambio(moeda, self.moeda_base)
        valor_convertido = round(valor_original * taxa, 2)

        diferenca = abs(valor_convertido - valor_recebido)
        if valor_recebido == 0.0:
            status = "PENDING"
        elif diferenca <= 5.00:
            status = "PAID_CONCILIATED"
        else:
            status = "DIVERGENT"

        fatura = {
            "invoice_id": id_fatura,
            "client": cliente,
            "original_amount": valor_original,
            "currency": moeda.upper(),
            "exchange_rate": taxa,
            "converted_amount_brl": valor_convertido,
            "received_amount_brl": valor_recebido,
            "status": status,
            "processed_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        }

        self.faturas.append(fatura)

        # Pausa de 1 segundo entre faturas
        time.sleep(1.0)
        return fatura

    def gerar_relatorio_excel(self, nome_arquivo: str = "conciliacao_faturas.xlsx"):
        df = pd.DataFrame(self.faturas)
        df.to_excel(nome_arquivo, index=False)
        print(f"\n✅ Relatório exportado com sucesso: '{nome_arquivo}'")
        return df

# ==========================================
# 3. TESTE PRÁTICO DA ESTEIRA
# ==========================================
if __name__ == "__main__":
    esteira = EsteiraFaturamento(moeda_base="BRL")

    # Exemplo 1: USD
    taxa_usd = obter_taxa_cambio("USD", "BRL")
    valor_brl_esperado = 1500.00 * taxa_usd
    esteira.adicionar_fatura("INV-001", "Acme Corp (USA)", 1500.00, "USD", valor_recebido=valor_brl_esperado)

    # Exemplo 2: EUR
    esteira.adicionar_fatura("INV-002", "TechEurope (EUR)", 800.00, "EUR", valor_recebido=4000.00)

    # Exemplo 3: GBP
    esteira.adicionar_fatura("INV-003", "Global London Ltd", 1200.00, "GBP", valor_recebido=0.0)

    # Exibição do Resultado
    df_resultado = esteira.gerar_relatorio_excel()
    print("\n--- RESUMO DA ESTEIRA DE FATURAMENTO ---")
    print(df_resultado[["invoice_id", "currency", "original_amount", "exchange_rate", "converted_amount_brl", "status"]])

⚠️ Erro ao buscar câmbio para USD-BRL: HTTPSConnectionPool(host='economia.awesomeapi.com.br', port=443): Max retries exceeded with url: /last/USD-BRL (Caused by ResponseError('too many 429 error responses')). Usando taxa fallback 1.0.
⚠️ Erro ao buscar câmbio para USD-BRL: HTTPSConnectionPool(host='economia.awesomeapi.com.br', port=443): Max retries exceeded with url: /last/USD-BRL (Caused by ResponseError('too many 429 error responses')). Usando taxa fallback 1.0.
⚠️ Erro ao buscar câmbio para EUR-BRL: HTTPSConnectionPool(host='economia.awesomeapi.com.br', port=443): Max retries exceeded with url: /last/EUR-BRL (Caused by ResponseError('too many 429 error responses')). Usando taxa fallback 1.0.
⚠️ Erro ao buscar câmbio para GBP-BRL: HTTPSConnectionPool(host='economia.awesomeapi.com.br', port=443): Max retries exceeded with url: /last/GBP-BRL (Caused by ResponseError('too many 429 error responses')). Usando taxa fallback 1.0.

✅ Relatório exportado com sucesso: 'conciliacao_faturas.xls